In [19]:
import torch
from torch.utils.data import DataLoader
import numpy as np
import json
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from dataset import create_dataloaders

# Keypoint indices
LEFT_SHOULDER = 5
RIGHT_SHOULDER = 6
LEFT_HIP = 11
RIGHT_HIP = 12

# Per-frame thresholds (coordinates are normalised)
HIP_VELOCITY_THRESHOLD = 0.005 # Rule 1: hip descent over VELOCITY_WINDOW frames
BODY_HORIZONTAL_THRESHOLD = 0.1 # Rule 2: shoulder to hip vertical gap
RATIO_THRESHOLD = 1.25 # Rule 3: body width-to-height ratio

# Full-sequence parameters
VELOCITY_WINDOW = 5 # Number of frames to look back for Rule 1
FRAME_VOTE_THRESHOLD = 5 # Minimum classified frames to classify sequence as fall
MIN_RULES_PER_FRAME = 2 # Minimum rules that must be called in a frame to classify it


def classify_frame(sequence, t):
    frame = sequence[t]

    # Calculate hip and shoulder midpoint Y coordinate at current frame
    hip_y = (frame[LEFT_HIP, 1] + frame[RIGHT_HIP, 1]) / 2
    shoulder_y = (frame[LEFT_SHOULDER, 1] + frame[RIGHT_SHOULDER, 1]) / 2

    # Rule 1: Hip descent over the a number of frames
    if t >= VELOCITY_WINDOW:
        prior_frame = sequence[t - VELOCITY_WINDOW]
        prior_hip_y = (prior_frame[LEFT_HIP, 1] + prior_frame[RIGHT_HIP, 1]) / 2
        rule1 = (hip_y - prior_hip_y) > HIP_VELOCITY_THRESHOLD
    else:
        rule1 = False # Not enough prior frames

    # Rule 2: Body orientation (check vertical gap between hips and shoulders)
    rule2 = abs(shoulder_y - hip_y) < BODY_HORIZONTAL_THRESHOLD

    # Rule 3: Width-to-height ratio of skeleton bounding box
    all_x = frame[:, 0]
    all_y = frame[:, 1]
    width  = all_x.max() - all_x.min()
    height = all_y.max() - all_y.min()

    # Protect against bad frames (e.g. all-zero keypoints)
    if height > 0:
        rule3 = (width / height) > RATIO_THRESHOLD
    else:
        rule3 = False

    rules_called = [r for r, called in enumerate([rule1, rule2, rule3], start=1) if called]
    return rules_called


def classify_sequence(sequence):
    num_frames = sequence.shape[0]
    fall_frame_count = 0
    rule_call_counts = {1: 0, 2: 0, 3: 0}

    # Iterate through frames
    for t in range(num_frames):
        rules_called = classify_frame(sequence, t)

        for r in rules_called:
            rule_call_counts[r] += 1

        if len(rules_called) >= MIN_RULES_PER_FRAME:
            fall_frame_count += 1

    # If enough frames are classified, classify the sequence as a fall
    prediction = 1 if fall_frame_count >= FRAME_VOTE_THRESHOLD else 0
    return prediction, fall_frame_count, rule_call_counts


def evaluate(dataloader, split_name="Validation"):
    all_preds = []
    all_labels = []

    total_rule_calls = {1: 0, 2: 0, 3: 0}
    total_fall_frames = 0

    # Iterate through batches
    for batch in dataloader:
        keypoints = batch['keypoints'].cpu().numpy()
        labels = batch['label'].cpu().numpy()

        # Classify frames
        for i in range(len(labels)):
            pred, fall_frame_count, rule_call_counts = classify_sequence(keypoints[i])
            all_preds.append(pred)
            all_labels.append(labels[i])
            total_fall_frames += fall_frame_count
            for r in rule_call_counts:
                total_rule_calls[r] += rule_call_counts[r]

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    n_clips = len(all_labels)
    n_frames = n_clips * 60

    print(f"\n{'=' * 55}")
    print(f"  Rule-Based Baseline (Per-Frame) — {split_name} Set")
    print(f"{'=' * 55}")
    print(f"\nParameters:")
    print(f"  HIP_VELOCITY_THRESHOLD:    {HIP_VELOCITY_THRESHOLD}")
    print(f"  BODY_HORIZONTAL_THRESHOLD: {BODY_HORIZONTAL_THRESHOLD}")
    print(f"  RATIO_THRESHOLD:           {RATIO_THRESHOLD}")
    print(f"  VELOCITY_WINDOW:           {VELOCITY_WINDOW} frames")
    print(f"  MIN_RULES_PER_FRAME:       {MIN_RULES_PER_FRAME}")
    print(f"  FRAME_VOTE_THRESHOLD:      {FRAME_VOTE_THRESHOLD} frames")

    print(f"\nRule called rates (across {n_frames} total frames):")
    for rule, count in total_rule_calls.items():
        print(f"  Rule {rule}: called in {count} frames ({count / n_frames * 100:.1f}%)")

    print(f"\nAverage fall-voting frames per clip: {total_fall_frames / n_clips:.1f} / 60")

    print(f"\n{classification_report(all_labels, all_preds, target_names=['No Fall', 'Fall'], digits=4)}")

    cm = confusion_matrix(all_labels, all_preds)
    tn, fp, fn, tp = cm.ravel()
    print(f"Confusion Matrix:")
    print(f"  TN={tn}  FP={fp}")
    print(f"  FN={fn}  TP={tp}")

    f1 = f1_score(all_labels, all_preds)
    print(f"\nOverall F1 (fall class): {f1:.4f}")

    return all_preds, all_labels


# MAIN
# Load dataloaders
_, val_loader, _ = create_dataloaders(
    split_file='data/dataset_split.json',
    batch_size=32,
    num_workers=4,
    normalise=True
)

# Evaluate on validation set to tune thresholds during development
evaluate(val_loader, split_name="Validation")


Loaded train set:
  Total clips: 5019
  Fall clips: 2280
  No-fall clips: 2739
  Resolution distribution:
    1920x1080: 2353 clips (46.9%)
    1100x1080: 2193 clips (43.7%)
    1280x720: 150 clips (3.0%)
    3840x2160: 130 clips (2.6%)
    848x480: 53 clips (1.1%)
    ... and 10 more resolutions
  Using per-video normalization

Loaded val set:
  Total clips: 1106
  Fall clips: 474
  No-fall clips: 632
  Resolution distribution:
    1920x1080: 523 clips (47.3%)
    1100x1080: 465 clips (42.0%)
    1280x720: 43 clips (3.9%)
    3840x2160: 19 clips (1.7%)
    2336x1080: 13 clips (1.2%)
    ... and 8 more resolutions
  Using per-video normalization

Loaded test set:
  Total clips: 1073
  Fall clips: 457
  No-fall clips: 616
  Resolution distribution:
    1920x1080: 503 clips (46.9%)
    1100x1080: 472 clips (44.0%)
    1280x720: 39 clips (3.6%)
    3840x2160: 22 clips (2.1%)
    848x480: 12 clips (1.1%)
    ... and 6 more resolutions
  Using per-video normalization

  Rule-Based Baseline